# 04 - NETTOYAGE DES MESURES ET SAUVEGARDE DANS UNE TABLE SILVER

Dans ce notebook, nous allons nettoyer les mesures en enlevant les lignes aberrantes ou lignes possédant des données manquantes sur les colonnes essentielles.

Ce nettoyage nous permets de standardiser les données et de les enregistrer dans la zone silver avec la table `silver_measures`.

In [9]:
# Chargement des librairies
import requests
import pyspark.sql.functions as F
import tarfile
from notebookutils import mssparkutils # La gestion du lakehouse
import os
from pathlib import Path

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 11, Finished, Available, Finished, False)

## Lecture de la table brute `raw_measures`

In [10]:
RAW_TABLE_MEASURES = "abfss://0bcbec59-dbbb-41e1-a6a0-d3763e6c43e6@onelake.dfs.fabric.microsoft.com/23840c0a-00b7-4a0b-af79-896b6c9be935/Tables/raw/radiation_measures"

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 12, Finished, Available, Finished, False)

In [11]:
# Lecture de la table brute à nettoyer
df_measures = spark.read.format("delta").load(RAW_TABLE_MEASURES)

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 13, Finished, Available, Finished, False)

## Application du nettoyage des mesures

In [12]:
prepared_measures = (df_measures
                                # Conversion des types de colonnes
                                .withColumns(
                                {
                                    "value" : df_measures.value.try_cast("float"),
                                    "startTime" : F.to_timestamp("startTime", "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                                    "endTime" : F.to_timestamp("endTime", "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
                                    "endLatitude" : F.col("endLatitude").cast("float"),
                                    "endLongitude" : F.col("endLongitude").cast('float'),
                                    "endAltitude" : F.col("endAltitude").cast('float'),
                                    "endAccuracy" : F.col("endAccuracy").cast('float')
                                }
                                )
                                # Filtrer pour enlever les valeurs manquantes
                                .filter("""reportUuid IS NOT NULL 
                                AND dateAndTimeOfCreation IS NOT NULL
                                AND value IS NOT NULL
                                AND startTime IS NOT NULL
                                AND endTime IS NOT NULL
                                AND latitude IS NOT NULL
                                AND longitude IS NOT NULL
                                """)

                                # Filtre pour enlever les valeurs aberrantes
                                .filter(
                                    """
                                    value > 0
                                    AND startTime <= endTime
                                    """
                                )

                                # Filtre pour que les mesures soient seulement avant la date d'aujourd'hui
                                .filter(
                                    """
                                    endTime <= date_sub(current_date(), 1)
                                    """
                                )
                                # Mettre valeur de début de position à la valeur de fin de position si nulle
                                # Nettoyage de la chaîne de caractère sur les identifiants d'appareils
                                .withColumns(
                                    {
                                        "endAltitude" : F.ifnull("endAltitude", "altitude"),
                                        "endLatitude" : F.ifnull("endLatitude", "latitude"),
                                        "endLongitude" : F.ifnull("endLongitude", "longitude"),
                                        "endAccuracy" : F.ifnull("endAccuracy", "accuracy"),
                                        "apparatusId" : F.when(F.trim(F.col("apparatusId")) == "", None).otherwise(F.lower(F.trim(F.col("apparatusId")))),
                                        "apparatusVersion" : F.when(F.trim(F.col("apparatusVersion")) == "", None).otherwise(F.lower(F.trim(F.col("apparatusVersion")))),
                                        "apparatusSensorType" : F.when(F.trim(F.col("apparatusSensorType")) == "", None).otherwise(F.lower(F.trim(F.col("apparatusSensorType")))),
                                        "apparatusTubeType" : F.when(F.trim(F.col("apparatusTubeType")) == "", None).otherwise(F.lower(F.trim(F.col("apparatusTubeType")))),
                                        "userId" :  F.when(F.trim(F.col("userId")) == "", None).otherwise(F.lower(F.trim(F.col("userId"))))
                                    }
                                )
                                # Sélectionner les colonnes pour une version fixe
                                .select(
                                    "reportUuid",
                                    "apparatusId",
                                    "apparatusVersion",
                                    "apparatusSensorType",
                                    "apparatusTubeType",
                                    "value",
                                    "hitsNumber",
                                    "calibrationFunction",
                                    "startTime",
                                    "endTime",
                                    "latitude",
                                    "longitude",
                                    "accuracy",
                                    "altitude",
                                    "endLatitude",
                                    "endLongitude",
                                    "endAltitude",
                                    "endAccuracy",
                                    "manualReporting",
                                    "organisationReporting",
                                    "description",
                                    "userId",
                                    "measurementEnvironment",
                                    "departureTime",
                                    "arrivalTime",
                                    "airportOrigin",
                                    "airportDestination",
                                    "aircraftType",
                                    "firstLatitude",
                                    "firstLongitude",
                                    "midLatitude",
                                    "midLongitude",
                                    "lastLatitude",
                                    "lastLongitude",
                                    "dateAndTimeOfCreation",
                                    "reliability",
                                    "qualification",
                                    "atypical"
                                )
                            )

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 14, Finished, Available, Finished, False)

## Ajout de la durée de mesure

In [13]:
prepared_measures_dury = (prepared_measures
        .withColumns({"dury" : F.timestamp_diff("SECOND", "startTime", "endTime")})
        )

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 15, Finished, Available, Finished, False)

In [14]:
prepared_measures_dury.printSchema()

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 16, Finished, Available, Finished, False)

root
 |-- reportUuid: string (nullable = true)
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- value: float (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- endTime: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAltitude: double (nullable = true)
 |-- endAccuracy: double (nullable = true)
 |-- manualReporting: double (nullable = true)
 |-- organisationReporting: string (nullable = true)
 |-- description: string (nullable = true)
 |-- userId: string (nullable = true)
 |-- measurementEnvironment: str

## Enregistrement dans le lakehouse dans la table silver `silver_measures`

In [15]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 17, Finished, Available, Finished, False)

DataFrame[]

In [16]:
prepared_measures_dury.write.format("delta").mode("overwrite").saveAsTable("silver.clean_measures")

StatementMeta(, c9b4a4e9-f9d8-47f6-af13-ba1b2702217c, 18, Finished, Available, Finished, True)